In [ ]:
# 01_free_particle.ipynb
# Simulation of a free particle in 1D
# using the split-operator algorithm implemented in Qiskit
# Reference: Kassal et al., PNAS 2008, arXiv:0801.2986

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [ ]:
# Grid parameters
n_qubits = 4
N = 2**n_qubits
L = 20.0
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)

In [ ]:
# Momentum grid (DFT order)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N) * N * dp
#print(p)

In [ ]:
# Initial wavepacket parameters
x0 = -5.0
sigma = 1.0
p0 = 1.0

# Gaussian wavepacket
psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * p0 * x)

# Normalize
psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

In [ ]:
# Plot initial wavepacket
plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi)**2)
plt.xlabel("x")
plt.ylabel("|psi(x)|^2")
plt.title("Initial wavepacket")
plt.grid(True)
plt.show()

In [ ]:
# Time parameters
dt = 0.05
n_steps = 40

# Kinetic propagator phases
kinetic_phases = np.exp(-1j * p**2 / 2 * dt)

In [ ]:
# Time evolution with classical split-operator (FFT)
psi_t = psi.copy()
positions = []

for step in range(n_steps):
    # Transform to momentum space
    psi_p = np.fft.fft(psi_t)
    
    # Apply kinetic propagator
    psi_p = psi_p * kinetic_phases
    
    # Transform back to position space
    psi_t = np.fft.ifft(psi_p)
    
    # Store expectation value of position
    x_mean = np.sum(x * np.abs(psi_t)**2) * dx
    positions.append(x_mean)

In [ ]:
# Analytical solution: <x(t)> = x0 + p0 * t
times = np.arange(1, n_steps + 1) * dt
x_analytical = x0 + p0 * times

# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, positions, label="Split-operator (FFT)")
plt.plot(times, x_analytical, "--", label="Analytical")
plt.xlabel("t")
plt.ylabel("<x>")
plt.title("Free particle: expectation value of position")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Check norm conservation
psi_t = psi.copy()
norms = []

for step in range(n_steps):
    psi_p = np.fft.fft(psi_t)
    psi_p = psi_p * kinetic_phases
    psi_t = np.fft.ifft(psi_p)
    norm = np.sum(np.abs(psi_t)**2) * dx
    norms.append(norm)

plt.figure(figsize=(10, 4))
plt.plot(times, norms)
plt.xlabel("t")
plt.ylabel("norm")
plt.title("Norm conservation")
plt.ylim(0.99, 1.01)
plt.grid(True)
plt.show()